# CTGAN

Second of the four generation notebooks and the first neural method. Same shape as the
previous one: load the shared training data, fit the generator, produce a synthetic
dataset of the same size, save it, and log the timings.

CTGAN trains two networks against each other. A generator produces fake records and a
discriminator tries to tell them from real ones, so as training progresses the generator
learns to produce records the discriminator can no longer distinguish. It was designed for
tables that mix numbers and categories, and for data where some categories are rare, both
of which describe this cohort.

Its known trade-off is training stability. Adversarial training can wobble, and the
quality of the result depends on the two networks improving at a similar pace.

## Setup

In [1]:
%pip install -q pandas pyarrow sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.9/209.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 144.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.0/207.0 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 11.4 MB/s eta 0:00:00


## Working folder

Sets the project folder so everything the pipeline writes, the cohort, the synthetic
datasets, the outputs and the figures, persists between sessions rather than sitting on
temporary storage.

The cohort and the synthetic datasets derive from MIMIC-IV, which is credentialed data
under a PhysioNet data use agreement. Keep the folder private, do not share it, and
delete the data once the work is finished.

In [2]:
import os
from pathlib import Path

# Use the shared project folder when one is available, otherwise stay in the
# current directory.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path("/content/drive/MyDrive/mimic-synthetic-pipeline")
    project_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(project_dir)
    print(f"Working folder: {project_dir}")
except ImportError:
    print("Using the local working folder.")

Mounted at /content/drive
Working folder set to Google Drive: /content/drive/MyDrive/mimic-synthetic-pipeline


## Hardware check

CTGAN is a neural network, so training time depends heavily on the hardware. On a GPU
this cohort trains in minutes; on a CPU it can take well over an hour. Worth knowing
before committing the time.

In [3]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "No GPU detected. CTGAN will train on CPU, which can take over an hour on this "
        "cohort. If a GPU was expected, enable it in the session settings, then "
        "restart the session and re-run from the beginning."
    )

CUDA available: True
GPU: NVIDIA L4


## Training data

In [4]:
from pathlib import Path

import pandas as pd

if not Path("data/train.parquet").exists():
    raise FileNotFoundError(
        "data/train.parquet not found. Run the extraction and preparation steps first."
    )

TARGET = "readmitted_30d"
train_df = pd.read_parquet("data/train.parquet")
print(f"Training data: {len(train_df):,} admissions, readmission rate {train_df[TARGET].mean():.4f}")

Training data: 427,408 admissions, readmission rate 0.2067


## Fitting and generating

Each method is trained to its own convergence rather than to a shared epoch count.
Matching epoch counts sounds fairer but is not, because an epoch means something different
for an adversarial game, a variational autoencoder and a diffusion model. Holding it
constant would measure which method converges fastest in epochs rather than which produces
the better synthetic data. Quality is compared at each method's own ceiling, and the cost
of reaching that ceiling is reported separately in the timing log.

CTGAN is the awkward case. Adversarial training has no principled convergence criterion,
because the generator is scored against a discriminator that is itself still improving, so
a falling loss does not indicate better samples. There is no equivalent of the loss
plateau test used for TVAE and TabDDPM. The approach here is a generous fixed budget, with
the absence of a stopping rule recorded as a known limitation of the method rather than
glossed over.

Checkpointing during training and keeping the epoch that scored best on fidelity was
rejected. With only a train and test split available, selecting on the test set would leak
model selection into the headline results.

In [5]:
import time

from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer

EPOCHS = 300  # Generous fixed budget. Adversarial training has no principled
              # convergence test, so this is set well above the 50-epoch feasibility
              # pass rather than tuned. At roughly 23 seconds per epoch on an L4 this
              # is about two hours.

metadata = Metadata.detect_from_dataframe(train_df, table_name="cohort")
model = CTGANSynthesizer(metadata, epochs=EPOCHS, verbose=True)

t0 = time.time()
model.fit(train_df)
train_seconds = time.time() - t0

t0 = time.time()
synthetic_df = model.sample(num_rows=len(train_df))
generate_seconds = time.time() - t0

synthetic_df.to_parquet("data/synthetic_ctgan.parquet", index=False)
print(f"Training took {train_seconds:.1f}s, generation took {generate_seconds:.1f}s")
print(f"Saved {len(synthetic_df):,} synthetic admissions to data/synthetic_ctgan.parquet")

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-00.27) | Discrim. (-00.11): 100%|██████████| 300/300 [1:50:18<00:00, 22.06s/it]


Training took 6724.9s, generation took 5.5s
Saved 427,408 synthetic admissions to data/synthetic_ctgan.parquet


## Training convergence

The epoch count is a quality lever and should be justified by evidence. This records the
loss curve so the choice can be defended, and saves both the chart and the raw values.

One caveat specific to CTGAN: adversarial losses are a stability diagnostic, not a
convergence test. The generator is scored against a discriminator that is itself still
learning, so the target moves and a falling generator loss does not mean the synthetic
data is improving. What the chart is genuinely useful for is spotting a run that has
diverged or collapsed, which shows up as one loss running away in magnitude. The epoch
count itself is chosen from the fidelity and utility results, not from this curve. TVAE
and TabDDPM, whose losses are ordinary likelihood-style objectives, can be read the usual
way.

In [ ]:
import matplotlib.pyplot as plt

fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

loss_df = model.get_loss_values()
# The library has shipped this column under two spellings ("Distriminator" is a
# long-standing typo upstream), so match whichever one is present.
disc_col = next(c for c in loss_df.columns if c.lower().startswith(("discrim", "distrim")))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(loss_df["Epoch"], loss_df["Generator Loss"], label="Generator", color="#4C72B0")
ax.plot(loss_df["Epoch"], loss_df[disc_col], label="Discriminator", color="#C44E52")
ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title(f"CTGAN adversarial losses over {EPOCHS} epochs")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "training_convergence_ctgan.png", dpi=150)
plt.show()

# Statistics are printed before anything is written to disk, so a file-writing problem
# cannot cost the diagnostic after a two-hour training run.
final = loss_df.tail(10)
print(
    f"Final 10 epochs, generator loss:     "
    f"mean {final['Generator Loss'].mean():+.3f}, sd {final['Generator Loss'].std():.3f}"
)
print(
    f"Final 10 epochs, discriminator loss: "
    f"mean {final[disc_col].mean():+.3f}, sd {final[disc_col].std():.3f}"
)
print(
    "\nRead this as a stability check only. Both losses oscillating within a bounded "
    "range is the healthy outcome; either one growing steadily in magnitude would "
    "indicate a diverging run. Do not infer the right epoch count from these curves, "
    "because the generator is scored against a moving target. Choose the epoch count "
    "from the fidelity and utility results in 07_evaluation.ipynb."
)

# Saving the raw values is a convenience, not a result, so it must not be able to stop
# this cell. Installing sdv can leave the session with a pandas whose internals are
# inconsistent, which surfaces as "AttributeError: 'Index' object has no attribute
# '_format_native_types'". Rebuilding the frame with plain string column names avoids
# that code path, and any remaining failure is caught and reported.
try:
    to_save = pd.DataFrame(loss_df.to_numpy(), columns=[str(c) for c in loss_df.columns])
    to_save.to_csv(out_dir / "loss_history_ctgan.csv", index=False)
    print(f"\nSaved loss history to {out_dir / 'loss_history_ctgan.csv'}")
except Exception as exc:  # noqa: BLE001
    print(f"\nCould not write the loss history CSV ({type(exc).__name__}: {exc}).")
    print("The statistics above are unaffected. Restarting the runtime clears this.")

## Sanity checks

The same quick aggregate checks as the other generation notebooks. The readmission rate
and the two numeric averages should be close to the real training values.

In [7]:
checks = pd.DataFrame({
    "statistic": ["Readmission rate", "Mean age", "Mean length of stay (days)"],
    "real_training_data": [
        round(train_df[TARGET].mean(), 4),
        round(train_df["age_at_admission"].astype(float).mean(), 1),
        round(train_df["length_of_stay_days"].mean(), 2),
    ],
    "synthetic_data": [
        round(synthetic_df[TARGET].mean(), 4),
        round(synthetic_df["age_at_admission"].astype(float).mean(), 1),
        round(synthetic_df["length_of_stay_days"].mean(), 2),
    ],
})
checks

,statistic,real_training_data,synthetic_data
0,Readmission rate,0.2067,0.2168
1,Mean age,58.8000,59.5000
2,Mean length of stay (days),4.6400,4.6400


In [8]:
# Append this run's timings to the shared generation log used by all four methods.
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
log_path = out_dir / "generation_log.csv"

entry = pd.DataFrame([{
    "method": "CTGAN",
    "rows_generated": len(synthetic_df),
    "train_seconds": round(train_seconds, 1),
    "generate_seconds": round(generate_seconds, 1),
}])
if log_path.exists():
    log = pd.read_csv(log_path)
    log = log[log["method"] != "CTGAN"]
    log = pd.concat([log, entry], ignore_index=True)
else:
    log = entry
log.to_csv(log_path, index=False)
log

,method,rows_generated,train_seconds,generate_seconds
0,TVAE,427408,453.1,3.1
1,Gaussian Copula,427408,105.3,7.3
2,TabDDPM,427408,6153.8,83.4
3,CTGAN,427408,6724.9,5.5
